# From transcripts to speaker turns

The saved October 2026 run yielded 200 calls and 8,792 turns. Six calls were flagged for Q&A review.

**Running in Google Colab.** Set `save_folder` to a writable Drive folder. The collection cells read the current Motley Fool transcript directory. Its contents change over time. Run the cells in order.

In [ ]:
%pip -q install requests beautifulsoup4 pandas

import requests
import pandas as pd
import re
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from IPython.display import display

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

save_folder = Path("/content/drive/MyDrive/TextMining/EarningsCalls")
save_folder.mkdir(parents=True, exist_ok=True)

print("Save folder:", save_folder)

In [ ]:
foolSite = "https://www.fool.com/earnings-call-transcripts/"

callSites = []
callSites.append(foolSite)

for i in range(9):
    callSites.append(foolSite + "page/" + str(i + 2) + "/")

callSites

In [ ]:
callLinks = []

for page_number, url in enumerate(callSites, start=1):
    response = requests.get(url, timeout=45)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    atags = soup.find_all("a", href=True)

    page_links = []

    for a in atags:
        link = urljoin(foolSite, a["href"])
        link = link.split("?")[0].split("#")[0]

        if re.search(r"/earnings/call-transcripts/\d{4}/", link):
            if link not in page_links:
                page_links.append(link)

    print(f"Page {page_number}: {len(page_links)} calls")

    if len(page_links) != 20:
        raise ValueError(
            f"Page {page_number} returned {len(page_links)} links; "
            "check the page before continuing."
        )

    callLinks.extend(page_links)
    time.sleep(1)

callLinks = list(dict.fromkeys(callLinks))

print("\nTotal unique calls:", len(callLinks))

if len(callLinks) != 200:
    raise ValueError("Expected 200 unique calls. Check for overlapping pages.")

In [ ]:
import re
import pandas as pd
from bs4 import BeautifulSoup

speakerRegEx = re.compile(r'^([^:]{1,100}):\s*(.*)$', re.S)
qaHeadingRegEx = re.compile(
    r'^(?:questions?\s*(?:and|&|-and-)\s*answers?|q\s*&\s*a)'
    r'(?:\s+(?:session|period))?\s*:?\s*$', re.I
)
qaRegEx = re.compile(
    r'\bfirst questions?\b|'
    r'\b(?:go|turn) first (?:this morning |today )?to\b|'
    r'\b(?:now,?\s+)?this question comes from\b|'
    r'\bno questions in (?:the )?queue\b|'
    r'\b(?:we will|we shall|we\'ll|we are|we\'re|let us|let\'s)\s+'
    r'(?:now\s+)?(?:be\s+)?(?:begin|start|conduct|conducting|move on to|'
    r'proceed (?:with|to)|ready to (?:take|take in|answer))\b'
    r'[^.!?]{0,90}\b(?:questions?|q\s*&\s*a)\b|'
    r'\bopen\s+(?:up\s+)?(?:the\s+)?(?:call|floor|lines?)'
    r'\s+(?:up\s+)?(?:for|to)\s+(?:questions|q\s*&\s*a)\b',
    re.I
)
hostQARegEx = re.compile(
    r'\b(?:the|our|your) first questions?\b[^.!?]{0,65}'
    r'\b(?:comes? from|is(?: going to come)? from|relates to)\b|'
    r'\b(?:the|our|your) first questions? is,\s|'
    r'\bturning to questions now\b', re.I
)
handoffRegEx = re.compile(
    r'(?:with that|we will now|we\'ll now|let\'s|we are now ready to)'
    r'[^.!?]{0,70}\b(?:open|begin)\b[^.!?]{0,50}'
    r'\b(?:questions|q\s*&\s*a)\b[^.!?]*[.!?]?\s*'
    r'(?:Operator,?\s*please begin[.!?]?)?\s*$', re.I
)
embeddedRegEx = re.compile(r'(Speaker\s+\d+)\s*(?=\[Operator Instructions\])')
dateRegEx = re.compile(
    r'\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|'
    r'Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Sept|'
    r'Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
    r'\.?\s+\d{1,2},?\s+\d{4}\b', re.I
)

def parse_call(html, call_id, diagnostics=False):
    soup = BeautifulSoup(html, 'html.parser')
    body = soup.select_one('#article-body-transcript, .article-body')
    if body is None or soup.h1 is None:
        raise ValueError('Transcript body or title not found')
    company = re.split(r'\s+\(|\s+Q[1-4]\s+', soup.h1.get_text(' ', strip=True), maxsplit=1)[0].strip()
    date_heading = body.find(['h2','h3','h4'], string=re.compile(r'^\s*DATE\s*$', re.I))
    date_text = date_heading.find_next('p').get_text(' ', strip=True) if date_heading else body.get_text(' ', strip=True)[:2500]
    match = dateRegEx.search(date_text)
    if not match:
        raise ValueError('Call date not found')
    date_text = re.sub(r'\bSept\b', 'Sep', match.group().replace('.', ''), flags=re.I)
    call_date = pd.to_datetime(date_text, errors='raise').strftime('%Y-%m-%d')
    for tag in body.select('.article-body-promobox, .pitch-container, script, style, aside, figure'):
        tag.decompose()
    blocks = body.find_all(['h2','h3','h4','p'])
    start = None
    for i, block in enumerate(blocks):
        label = block.get_text(' ', strip=True).lower().rstrip(':')
        if label in ['full conference call transcript', 'prepared remarks']:
            start = i + 1
            break
    if start is None:
        raise ValueError('Transcript starting point not found')
    turns = []
    full_text = []
    section = 'Prepared'
    qa_found = False
    boundary = ''
    boundary_index = None
    suspicious = []
    for block in blocks[start:]:
        text = block.get_text(' ', strip=True)
        if not text: continue
        if qaHeadingRegEx.search(text):
            section = 'QA'
            qa_found = True
            boundary = text
            boundary_index = len(turns)
            full_text.append(text)
            continue
        if block.name != 'p':
            if re.match(r'^(?:call participants|related articles|read more|more .*)\s*:?$', text, re.I):
                break
            continue
        if text.lower().startswith(('this article is a transcript', 'this transcript was', 'the motley fool has', 'the motley fool recommends')):
            break
        full_text.append(text)
        embedded = embeddedRegEx.search(text)
        if embedded:
            if not turns: raise ValueError('Embedded speaker before first turn')
            turns[-1]['Text'] += '\n\n' + text[:embedded.start()].strip()
            turns.append({'EarningsCallID':call_id,'SpeakerName':embedded.group(1),
                          'PreparedOrQA':section,'Text':text[embedded.end():].strip()})
            continue
        speaker = None
        speech = text
        bold = block.find(['strong', 'b'])
        if bold:
            prefix = bold.get_text(' ', strip=True)
            if text.startswith(prefix):
                match = speakerRegEx.search(text)
                if match and (':' in prefix or text[len(prefix):].startswith(':')):
                    speaker = match.group(1).strip()
                    speech = match.group(2).strip()
                elif re.search(r'\s(?:--|—|–)\s', text):
                    speaker = re.split(r'\s*(?:--|—|–)\s*', prefix)[0].rstrip(':').strip()
                    speech = text[len(prefix):].lstrip(' :')
                elif len(text) < 100 and text == prefix and re.fullmatch(r"[A-Z][\w .,'’()/-]+",text):
                    speaker = text
                    speech = ''
                else:
                    suspicious.append(text[:140])
        if speaker:
            turns.append({'EarningsCallID':call_id, 'SpeakerName':speaker,'PreparedOrQA':section, 'Text':speech})
        elif turns:
            turns[-1]['Text'] += '\n\n' + speech
        else:
            raise ValueError('Text found before first recognized speaker')
    turns = [turn for turn in turns if turn['Text'].strip()]
    if not turns:
        raise ValueError('No speaker turns found')
    if not qa_found:
        for i, turn in enumerate(turns):
            is_operator = turn['SpeakerName'].lower() in ['operator','conference operator'] or re.fullmatch(r'Speaker\s+\d+', turn['SpeakerName'])
            if i > 0 and ((is_operator and qaRegEx.search(turn['Text'])) or hostQARegEx.search(turn['Text'])):
                qa_found=True
                boundary=turn['Text']
                boundary_index=i
                for row in turns[i:]:row['PreparedOrQA']='QA'
                break
            handoff = handoffRegEx.search(turn['Text'][-450:])
            if i > 0 and i+1 < len(turns) and handoff and not re.search(r'\b(?:before|after|following)\b', handoff.group(), re.I):
                qa_found=True
                boundary=turn['Text'][-450:]
                boundary_index=i+1
                for row in turns[i+1:]:row['PreparedOrQA']='QA'
                break
    call={'EarningsCallID':call_id,'CompanyName':company,'CallDate':call_date,'EarningsCallText':'\n\n'.join(full_text)}
    if diagnostics:
        return call,turns,qa_found,{'boundary':boundary,'boundary_index':boundary_index,'suspicious':suspicious}
    return call,turns,qa_found

In [ ]:
html_folder = save_folder / "html"
html_folder.mkdir(exist_ok=True)

call_rows = []
turn_rows = []
errors = []
review_calls = []

for i, url in enumerate(callLinks, start=1):
    call_id = f"CALL{i:03d}"
    html_file = html_folder / f"{call_id}.html"

    try:
        if html_file.exists():
            html = html_file.read_text(encoding="utf-8")
        else:
            response = requests.get(url, timeout=45)
            response.raise_for_status()

            html = response.text
            html_file.write_text(html, encoding="utf-8")
            time.sleep(1)

        call, turns, qa_found = parse_call(html, call_id)

        call_rows.append(call)
        turn_rows.extend(turns)

        if not qa_found:
            review_calls.append({
                "EarningsCallID": call_id,
                "URL": url,
                "Reason": "No Q&A boundary detected"
            })

        print(f"{i}/200: {call['CompanyName']} — {len(turns)} turns")

    except Exception as e:
        errors.append({
            "EarningsCallID": call_id,
            "URL": url,
            "Error": str(e)
        })
        print(f"{i}/200: FAILED — {e}")

In [ ]:
DF1 = pd.DataFrame(
    call_rows,
    columns=[
        "EarningsCallID",
        "CompanyName",
        "CallDate",
        "EarningsCallText"
    ]
)

DF2 = pd.DataFrame(
    turn_rows,
    columns=[
        "EarningsCallID",
        "SpeakerName",
        "PreparedOrQA",
        "Text"
    ]
)

DF2.insert(
    0,
    "SpeakerTurnID",
    [f"TURN{i:06d}" for i in range(1, len(DF2) + 1)]
)

print("DF1:", DF1.shape)
print("DF2:", DF2.shape)
print("Failed calls:", len(errors))
print("Calls needing Q&A review:", len(review_calls))

display(DF1.head())
display(DF2.head(10))

display(
    DF2.groupby(["EarningsCallID", "PreparedOrQA"])
       .size()
       .unstack(fill_value=0)
)

if errors:
    display(pd.DataFrame(errors))

if review_calls:
    display(pd.DataFrame(review_calls))

if len(DF1) == 200 and not review_calls:
    print("200 calls processed. Inspect a few calls to check the parsing.")
else:
    print("Check the failed calls and Q&A review list before submitting.")

In [ ]:
DF1.to_csv(
    save_folder / "DF1.csv",
    index=False,
    encoding="utf-8-sig"
)

DF2.to_csv(
    save_folder / "DF2.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(
    errors,
    columns=["EarningsCallID", "URL", "Error"]
).to_csv(save_folder / "errors.csv", index=False)

pd.DataFrame(
    review_calls,
    columns=["EarningsCallID", "URL", "Reason"]
).to_csv(save_folder / "review_calls.csv", index=False)

pd.DataFrame({
    "EarningsCallID": [
        f"CALL{i:03d}" for i in range(1, len(callLinks) + 1)
    ],
    "URL": callLinks
}).to_csv(save_folder / "call_urls.csv", index=False)

print("Saved to:", save_folder)

if not DF1.empty:
    selected_call = DF1.iloc[0]["EarningsCallID"]

    for _, row in DF2[DF2["EarningsCallID"] == selected_call].head(8).iterrows():
        print("\n", row["SpeakerTurnID"],
              row["SpeakerName"], row["PreparedOrQA"])
        print(row["Text"][:600])